In [8]:
from pathlib import Path

project_root = Path("..").resolve()
exporter_path = project_root / "dashboard_exporter.py"

dashboard_code = '''from pathlib import Path
import copy
import inspect
import json
import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

from utils.model_factory import create_models
from utils.visualizer import Visualizer


@st.cache_resource(show_spinner=False)
def get_cached_resources():
    base_dir = Path(__file__).resolve().parent
    data_dir = base_dir / "data" / "processed"

    stops_data = pd.read_csv(
        data_dir / "bus_stops.csv",
        encoding="utf-8-sig",
        dtype={"stop_id": str, "district_code": str},
    )
    districts_data = gpd.read_file(data_dir / "districts.geojson")
    districts_data["district_code"] = districts_data["district_code"].astype(str)
    # 폴리곤 좌표 단순화로 브라우저 전송 용량 대폭 축소
    districts_data["geometry"] = districts_data["geometry"].simplify(tolerance=0.001, preserve_topology=True)
    geojson_dict = json.loads(districts_data.to_json())

    bus_stops, districts = create_models(stops_data, districts_data)
    viz = Visualizer(districts, bus_stops)
    map_df = viz.prepare_map_data()

    # 구별 정류소 데이터를 미리 딕셔너리로 인덱싱 (클릭 시 전체 순회 제거)
    stops_by_code = {}
    for s in bus_stops:
        c = str(getattr(s, "district_code", ""))
        stops_by_code.setdefault(c, []).append(s)

    # 기본 25개 구 전체 지도 1회만 사전 생성
    score_col = "equity_score" if "equity_score" in map_df.columns else map_df.select_dtypes(include="number").columns[0]
    base_map_fig = px.choropleth(
        map_df,
        geojson=geojson_dict,
        locations="district_code",
        featureidkey="properties.district_code",
        color=score_col,
        color_continuous_scale=getattr(viz, "COLORSCALE", "Viridis"),
        hover_name="district_name",
        custom_data=["district_code"],
    )
    base_map_fig.update_geos(fitbounds="locations", visible=False)
    base_map_fig.update_layout(height=500, margin=dict(l=0, r=0, t=10, b=0), clickmode="event+select")

    return viz, map_df, geojson_dict, stops_by_code, base_map_fig


@st.cache_data(show_spinner=False)
def get_cached_charts(_viz, target_code, target_name):
    """선택 상태별 막대그래프 및 OLS 산점도 결과 캐싱 (재클릭 시 즉시 응답)"""
    def _call(func):
        sig = inspect.signature(func)
        params = list(sig.parameters.keys())
        if not params:
            return func()
        first_param = params[0]
        val = target_code if "code" in first_param.lower() else target_name
        return func(**{first_param: val})

    bar_fig = _call(_viz.create_bar_chart)
    bar_fig.update_traces(selectedpoints=None)

    scatter_fig = _call(_viz.create_scatter_plot)
    scatter_fig.update_traces(selectedpoints=None)
    return bar_fig, scatter_fig


class InteractiveDashboardExporter:
    def __init__(self, visualizer: Visualizer, map_df: pd.DataFrame, geojson_dict: dict, stops_by_code: dict, base_map_fig: go.Figure):
        self.viz = visualizer
        self.map_df = map_df
        self.geojson = geojson_dict
        self.stops_by_code = stops_by_code
        self.base_map_fig = base_map_fig

    def render(self):
        st.set_page_config(page_title="서울시 버스 인프라 형평성 대시보드", layout="wide")
        st.title("🚌 서울시 자치구별 버스 인프라 형평성 통합 대시보드")

        if "selected_code" not in st.session_state:
            st.session_state.selected_code = None

        code_to_name = dict(zip(self.map_df["district_code"], self.map_df["district_name"]))
        name_to_code = {v: k for k, v in code_to_name.items()}

        # 상단 자치구 검색창
        options = ["전체 보기 (선택 해제)"] + sorted(list(name_to_code.keys()))
        current_idx = 0
        if st.session_state.selected_code in code_to_name:
            current_idx = options.index(code_to_name[st.session_state.selected_code])

        picked_name = st.selectbox("🔍 자치구 검색 (지도 및 하단 차트와 실시간 연동)", options, index=current_idx)
        new_code = None if picked_name == "전체 보기 (선택 해제)" else name_to_code[picked_name]
        if new_code != st.session_state.selected_code:
            st.session_state.selected_code = new_code
            st.rerun()

        target_code = st.session_state.selected_code
        target_name = code_to_name.get(target_code)

        # 상단 요약 지표 카드 (전체 현황 vs 선택 자치구 현황)
        total_districts = len(self.map_df)
        total_stops = len(self.viz.bus_stops)

        m_col1, m_col2 = st.columns(2)
        if target_code is None:
            m_col1.metric(label="분석 대상 자치구", value=f"{total_districts:,}개")
            m_col2.metric(label="서울시 전체 버스정류소", value=f"{total_stops:,}개")
        else:
            district_stop_count = len(self.stops_by_code.get(str(target_code), []))
            m_col1.metric(label="선택된 자치구", value=target_name)
            m_col2.metric(
                label=f"{target_name} 버스정류소 수",
                value=f"{district_stop_count:,}개",
                delta=f"전체 {total_stops:,}개 중 {district_stop_count / total_stops * 100:.1f}%",
                delta_color="off",
            )

        st.divider()

        # 상단 지도: 캐싱된 기본 지도 위에 선택 구 녹색 폴리곤 + 정류소 핀만 가볍게 추가
        map_fig = self._build_fast_map(target_code)
        map_event = st.plotly_chart(map_fig, use_container_width=True, on_select="rerun", key="main_map")

        if map_event and map_event.get("selection", {}).get("points"):
            pt = map_event["selection"]["points"][0]
            clicked_code = pt.get("location") or (pt.get("customdata")[0] if pt.get("customdata") else None)
            if clicked_code and str(clicked_code) in code_to_name and str(clicked_code) != target_code:
                st.session_state.selected_code = str(clicked_code)
                st.rerun()

        # 하단 차트: 캐싱된 결과 즉시 호출
        st.divider()
        st.subheader("📊 자치구별 형평성 순위 및 인구 대비 정류소 OLS 회귀 분석")
        bar_fig, scatter_fig = get_cached_charts(self.viz, target_code, target_name)

        col1, col2 = st.columns(2)
        with col1:
            bar_event = st.plotly_chart(bar_fig, use_container_width=True, on_select="rerun", key="bar_chart")
            if bar_event and bar_event.get("selection", {}).get("points"):
                pt = bar_event["selection"]["points"][0]
                clicked_name = pt.get("y") or pt.get("x")
                if clicked_name in name_to_code and name_to_code[clicked_name] != target_code:
                    st.session_state.selected_code = name_to_code[clicked_name]
                    st.rerun()

        with col2:
            scatter_event = st.plotly_chart(scatter_fig, use_container_width=True, on_select="rerun", key="scatter_chart")
            if scatter_event and scatter_event.get("selection", {}).get("points"):
                pt = scatter_event["selection"]["points"][0]
                clicked_text = pt.get("text") or pt.get("hovertext")
                if clicked_text in name_to_code and name_to_code[clicked_text] != target_code:
                    st.session_state.selected_code = name_to_code[clicked_text]
                    st.rerun()

    def _build_fast_map(self, target_code: str | None) -> go.Figure:
        fig = go.Figure(self.base_map_fig)
        if not target_code:
            return fig

        # 선택된 구의 GeoJSON 피처 1개만 추출해 전송 (전체 25개 구 GeoJSON 중복 전송 방지)
        single_features = [
            f for f in self.geojson["features"]
            if str(f.get("properties", {}).get("district_code", "")) == str(target_code)
        ]
        single_geojson = {"type": "FeatureCollection", "features": single_features}

        fig.add_trace(go.Choropleth(
            geojson=single_geojson,
            locations=[str(target_code)],
            featureidkey="properties.district_code",
            z=[1],
            colorscale=[[0, "#2ecc71"], [1, "#2ecc71"]],
            showscale=False,
            marker_line_width=3.5,
            marker_line_color="#145a32",
            hoverinfo="skip",
        ))

        target_stops = self.stops_by_code.get(str(target_code), [])
        if target_stops:
            fig.add_trace(go.Scattergeo(
                lat=[getattr(s, "lat", getattr(s, "latitude", None)) for s in target_stops],
                lon=[getattr(s, "lon", getattr(s, "longitude", None)) for s in target_stops],
                mode="markers",
                marker=dict(size=6, color="#e74c3c", line=dict(width=1, color="white")),
                text=[getattr(s, "stop_name", getattr(s, "name", "")) for s in target_stops],
                name="버스정류소",
            ))

        return fig


if __name__ == "__main__":
    viz, map_df, geojson_dict, stops_by_code, base_map_fig = get_cached_resources()
    dashboard = InteractiveDashboardExporter(viz, map_df, geojson_dict, stops_by_code, base_map_fig)
    dashboard.render()
'''

exporter_path.write_text(dashboard_code, encoding="utf-8")
print("✅ 지도 중복 직렬화 제거 및 차트 캐싱 최적화 완료!")

✅ 지도 중복 직렬화 제거 및 차트 캐싱 최적화 완료!


In [9]:
import subprocess, sys, time, webbrowser
from pathlib import Path

project_root = Path("..").resolve()
exporter_path = project_root / "dashboard_exporter.py"

# 1. streamlit 설치 여부 확인 및 미설치 시 자동 설치
try:
    import streamlit
except ImportError:
    print("⏳ streamlit 패키지 자동 설치 중 (약 10초 소요)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])

# 2. 서버 실행 및 에러 로그 캡처
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(exporter_path), "--server.port", "8501", "--server.headless", "true"],
    cwd=str(project_root),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
time.sleep(4)

if proc.poll() is not None:
    print("❌ 서버 실행 실패 로그:\n", proc.stderr.read())
else:
    webbrowser.open("http://localhost:8501")
    print("✅ 서버 정상 가동! 브라우저 주소창에 http://localhost:8501 을 입력해 보세요.")

✅ 서버 정상 가동! 브라우저 주소창에 http://localhost:8501 을 입력해 보세요.
